In [1]:
import os

### Write ```Dockerfile```

In [2]:
%%writefile Dockerfile

FROM python:3.9
    
# update package manager
RUN apt-get update

# update pip
RUN pip install --upgrade pip

# copy requirements
COPY requirements.txt .

# install dependencies
RUN pip install -r requirements.txt

# copy script into container
COPY script.py .

# run script when image is run
CMD ["python3", "script.py"]

Writing Dockerfile


### Write ```requirements.txt``` to local drive

In [3]:
%%writefile requirements.txt

pyarrow==9.0.0
fsspec==2022.10.0
s3fs==2022.10.0

pandas==1.2.4

Writing requirements.txt


### Write ```script.py``` to local drive

In [4]:
%%writefile script.py

import os
import pandas as pd
import numpy as np
from datetime import datetime
import math

# constants
str_project = '20231010-gen-xii'
str_task = '13_payload_parsing'
str_subtask = 'split_payloads'
int_n_requests_per_lambda = 100

# get today's date
str_date_today = datetime.today().strftime('%Y%m%d')

# import payloads
str_filename = 'df_payloads.gzip'
str_uri = f's3://{str_project}/{str_task}/days/{str_date_today}/payloads/{str_filename}'
df = pd.read_parquet(str_uri)

# keep only the most recent payload
df.drop_duplicates(
    subset=['ACCOUNTID'],
    keep='last', 
    inplace=True,
)

# sort
df.sort_values(by='REQUEST_DATETIME', ascending=True, inplace=True)

# get nrows
int_nrows = df.shape[0]

# divide by int_n_requests_per_lambda
int_n_lambdas = math.ceil(int_nrows / int_n_requests_per_lambda)

# create list to assign as new column
list_rows = list(np.tile(np.arange(1, int_n_lambdas+1), int_n_requests_per_lambda))

# make sure its the same length as df
list_rows = list_rows[:int_nrows]

# assign
df['rows'] = list_rows

# make df_idx.csv
list_rows = list(df['rows'].value_counts().index)
df_idx = pd.DataFrame({'row': list_rows})
df_idx.sort_values(by='row', ascending=True, inplace=True)

# save
str_filename = 'df_idx.csv'
str_uri = f's3://{str_project}/{str_task}/{str_filename}'
df_idx.to_csv(str_uri, index=False)

# subset and save
for int_row in df_idx['row']:
    # subset
    df_tmp = df[df['rows'] == int_row].copy()
    # save
    str_filename = f'df_rows_{int_row}.gzip'
    str_uri = f's3://{str_project}/{str_task}/days/{str_date_today}/{str_subtask}/{str_filename}'
    df_tmp.to_parquet(str_uri, compression='gzip')

Writing script.py


### Build and push to ECR

In [5]:
%%sh

# name the image
image=genxii-split-payloads

# build image
docker build -t ${image} .

# get region
region=$(aws configure get region)
region=${region:-us-west-2}

# get account
account=$(aws sts get-caller-identity --query Account --output text)

# get full name
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# get login command and execute it
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# create repository in ECR
aws ecr create-repository --repository-name "${image}" --image-scanning-configuration scanOnPush=true --image-tag-mutability MUTABLE

# tag image
docker tag  ${image} ${fullname}

# push image to ECR   
docker push ${fullname}

Sending build context to Docker daemon  27.65kB
Step 1/7 : FROM python:3.9
 ---> 4b15bb967077
Step 2/7 : RUN apt-get update
 ---> Using cache
 ---> c59ecdb5b59b
Step 3/7 : RUN pip install --upgrade pip
 ---> Using cache
 ---> 692397bbf273
Step 4/7 : COPY requirements.txt .
 ---> Using cache
 ---> b78591275a29
Step 5/7 : RUN pip install -r requirements.txt
 ---> Using cache
 ---> 5079183a4607
Step 6/7 : COPY script.py .
 ---> e88079097e58
Step 7/7 : CMD ["python3", "script.py"]
 ---> Running in 1e7f153ff0e6
Removing intermediate container 1e7f153ff0e6
 ---> 1c55817314a7
Successfully built 1c55817314a7
Successfully tagged genxii-split-payloads:latest


WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded



An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'genxii-split-payloads' already exists in the registry with id '836690756591'


The push refers to repository [836690756591.dkr.ecr.us-west-2.amazonaws.com/genxii-split-payloads]
90c9886c69ec: Preparing
0f4ef7184374: Preparing
ee7f84bcf4f0: Preparing
bf86e0cc03ba: Preparing
69b638ea12af: Preparing
afe28ac5c5d1: Preparing
f3b460831925: Preparing
20e2f78dadaf: Preparing
e077e19b6682: Preparing
21e1c4948146: Preparing
68866beb2ed2: Preparing
e6e2ab10dba6: Preparing
0238a1790324: Preparing
20e2f78dadaf: Waiting
e077e19b6682: Waiting
21e1c4948146: Waiting
68866beb2ed2: Waiting
e6e2ab10dba6: Waiting
0238a1790324: Waiting
afe28ac5c5d1: Waiting
f3b460831925: Waiting
69b638ea12af: Layer already exists
ee7f84bcf4f0: Layer already exists
0f4ef7184374: Layer already exists
bf86e0cc03ba: Layer already exists
afe28ac5c5d1: Layer already exists
20e2f78dadaf: Layer already exists
f3b460831925: Layer already exists
e077e19b6682: Layer already exists
21e1c4948146: Layer already exists
68866beb2ed2: Layer already exists
e6e2ab10dba6: Layer already exists
0238a1790324: Layer already 

### Clean-up

In [6]:
# rm files
for str_file in ['Dockerfile','requirements.txt','script.py']:
    try:
        os.remove(f'./{str_file}')
    except:
        pass